# QCHAN v4.0.0 — Scientific finalization and freeze preparation

This notebook does not decode audio, rebuild spectra, rebuild references, or recompute the four QCHAN analysis features. It verifies the completed R3 cohort candidate, applies the accepted G10 roles, adds participant-bootstrap and iteration-wise reference uncertainty, regenerates five audited figures from saved tables, and prepares the immutable measurement freeze.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT_OVERRIDE = None  # installer inserts the local project root
RUN_PACKAGE_TESTS = True
RUN_FINALIZATION = True
SCIENTIFIC_REVIEW_DECISION = "PENDING"  # installer sets ACCEPT_QCHAN_V400
SCIENTIFIC_REVIEWER = "Nevena Musikic"
SCIENTIFIC_REVIEW_RATIONALE = (
    "Post-cohort scientific audit completed. Reference-relative LTAS distance is retained as primary nonordinal; "
    "rolloff-95 deficit is retained as primary one-sided; high-band ratio deficit is secondary and non-independent; "
    "tilt steepening is exploratory and phenotype-sensitive. No QCHAN scalar, device identity, or standalone gate is approved."
)
PUBLISH_AND_FREEZE = False

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "04_QCHAN":
    candidate = Path.cwd() / "notebooks/02_feature_extraction" / "04_QCHAN"
    if candidate.exists():
        NOTEBOOK_DIR = candidate
PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE) if PROJECT_ROOT_OVERRIDE else NOTEBOOK_DIR.parents[1]
SOURCE_ROOT = PROJECT_ROOT / "outputs/reviewed" / "channel_device" / "qchan-v4.0.0-candidate"
FINAL_ROOT = PROJECT_ROOT / "outputs/reviewed" / "channel_device" / "qchan-v4.0.0"

for source_path in (PROJECT_ROOT / "src", PROJECT_ROOT / "src"):
    if str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

from paper1_qc_reviewed.qchan_v400_final import (
    ACCEPTANCE_TOKEN,
    analysis_values_equal,
    finalize_candidate,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source candidate: {SOURCE_ROOT}")
print(f"Final candidate: {FINAL_ROOT}")
print(f"Decision: {SCIENTIFIC_REVIEW_DECISION}")


In [ ]:
if RUN_PACKAGE_TESTS:
    test_paths = [
        PROJECT_ROOT / "tests" / "test_qchan_v400.py",
        PROJECT_ROOT / "tests" / "test_qchan_v400_cohort.py",
        PROJECT_ROOT / "tests" / "test_qchan_v400_final.py",
    ]
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *map(str, test_paths), "-q", "--disable-warnings"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError("Reviewed QCHAN tests failed")


In [ ]:
source_manifest_path = SOURCE_ROOT / "manifests" / "qchan_v400_cohort_candidate_manifest.json"
source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
required = {
    "cohort_extraction_completed": True,
    "cohort_evidence_complete": True,
    "recording_count": 519,
    "participant_count": 224,
    "spectrum_count": 519,
    "reference_ledger_count": 519,
    "reference_vintage_count": 1,
    "required_panels_complete": True,
    "gallery_bundle_count": 8,
    "cohort_hotfix_revision": "gallery-linked-source-r3",
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "device_identity_estimated": False,
}
for key, expected in required.items():
    observed = source_manifest.get(key)
    if observed != expected:
        raise RuntimeError(f"Source manifest mismatch for {key}: {observed!r}")
print("Completed QCHAN R3 cohort candidate verified.")


In [ ]:
provenance_files = [
    NOTEBOOK_DIR / "support/QCHAN_v400_FINAL_SCIENTIFIC_AUDIT.md",
    NOTEBOOK_DIR / "support/QCHAN_v400_FINAL_FEATURE_DECISIONS.csv",
    NOTEBOOK_DIR / "support/QCHAN_Family_Evaluation_Workbook_v1_0.docx",
    NOTEBOOK_DIR / "support/QCHAN_V4_0_0_FREEZE_CONTRACT.md",
    NOTEBOOK_DIR / "support/QCHAN_Validation_Checklist_v1_0.csv",
    NOTEBOOK_DIR / "support/QCHAN_Ten_Domain_Dashboard_v1_0.csv",
    NOTEBOOK_DIR / "support/QCHAN_Gate_Summary_FINAL_v1_0.csv",
    NOTEBOOK_DIR / "support/QCHAN_V400_FINALIZATION_IMPLEMENTATION_REPORT.md",
]

if RUN_FINALIZATION:
    final_manifest = finalize_candidate(
        source_root=SOURCE_ROOT,
        final_root=FINAL_ROOT,
        scientific_review_decision=SCIENTIFIC_REVIEW_DECISION,
        scientific_reviewer=SCIENTIFIC_REVIEWER,
        scientific_review_rationale=SCIENTIFIC_REVIEW_RATIONALE,
        provenance_files=provenance_files,
        repeat_bootstrap_iterations=5000,
    )
    print(json.dumps(final_manifest, indent=2))


In [ ]:
if RUN_FINALIZATION:
    source_features = pd.read_csv(SOURCE_ROOT / "tables" / "qchan_v400_analysis_features.csv")
    final_features = pd.read_csv(FINAL_ROOT / "tables" / "qchan_v400_analysis_features.csv")
    if not analysis_values_equal(source_features, final_features):
        raise RuntimeError("Finalization changed QCHAN analysis feature values")
    repeat_ci = pd.read_csv(FINAL_ROOT / "validation" / "qchan_v400_repeated_recording_persistence_with_ci.csv")
    rank_stability = pd.read_csv(FINAL_ROOT / "validation" / "qchan_v400_reference_bootstrap_rank_stability.csv")
    figure_index = pd.read_csv(FINAL_ROOT / "figures" / "qchan_v400_standardized_figure_index.csv")
    print(f"Numerical equivalence: True ({len(final_features)} recordings)")
    print(f"Participant-pair CI rows: {len(repeat_ci)}")
    print(f"Reference-bootstrap rank rows: {len(rank_stability)}")
    print(f"Applicable figure bundles: {(figure_index.panel != 'I').sum()}")
    print(f"Panel I: {figure_index.loc[figure_index.panel.eq('I'), 'selection_reason'].item()}")


In [ ]:
if RUN_FINALIZATION:
    final_manifest_path = FINAL_ROOT / "manifests" / "qchan_v400_final_candidate_manifest.json"
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    if SCIENTIFIC_REVIEW_DECISION == ACCEPTANCE_TOKEN and not final_manifest.get("freeze_allowed"):
        raise RuntimeError("Accepted candidate did not authorize atomic freeze")
    if PUBLISH_AND_FREEZE:
        raise RuntimeError("This notebook prepares but does not execute the atomic freeze")
    print("QCHAN v4.0.0 FINALIZATION COMPLETE")
    print("Candidate is ready for atomic freeze." if final_manifest.get("freeze_allowed") else "Scientific acceptance remains pending.")
